# Notebook 101 — Línea base: RAG con TF-IDF

Antes de montar infraestructura vectorial, conviene saber qué tan lejos llega lo simple.
Este notebook construye un RAG **completo y funcional** usando solo TF-IDF con SparkML —
las mismas herramientas que ya conoces de la clase de ML.

Esto no es un ejercicio de calentamiento. Es la línea base contra la cual vas a medir si el
índice vectorial del notebook 102 vale lo que cuesta. 

## Arquitectura del pipeline

```
pregunta -> [retrieve: TF-IDF + coseno] -> chunks -> [generate: LLM] -> respuesta + citas
```

Las tres piezas (retrieve, generate, cite) son las mismas que usará el agente final. Lo único
que cambia en el notebook 102 es la implementación del retrieve.

## 1. Constantes y dependencias

In [ ]:
CATALOG = "big_data_ii_2025"
SCHEMA = "spark_examples"
VOL = f"/Volumes/{CATALOG}/{SCHEMA}/agenteval"

T_CORPUS = f"{CATALOG}.{SCHEMA}.agenteval_corpus"
T_DEV = f"{CATALOG}.{SCHEMA}.agenteval_train_dev"
T_BASELINE = f"{CATALOG}.{SCHEMA}.agenteval_baseline_answers"

TFIDF_MODEL_PATH = f"{VOL}/models/tfidf"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

### Tamaño de la muestra

Free Edition está sujeta a una política de uso justo. Cada respuesta implica una llamada al
LLM, así que trabajamos sobre una muestra y no sobre las miles de preguntas de `train`.

El widget aparece arriba del notebook y puedes cambiarlo sin editar código. Para la
demostración en clase, 20–30 es suficiente para ver diferencias reales entre configuraciones.

In [ ]:
dbutils.widgets.text("n_eval", "20", "Número de preguntas a evaluar")
N_EVAL = int(dbutils.widgets.get("n_eval"))
print(f"Se evaluarán {N_EVAL} preguntas.")

## 2. Descubrir el modelo LLM disponible

No hardcodeamos un nombre de modelo. Los endpoints de Foundation Model APIs disponibles en
Free Edition cambian con el tiempo y no todos los modelos están habilitados en todas las
cuentas. La celda consulta qué hay realmente y elige el primero de una lista de preferencia.

Si ninguno está disponible, el `assert` te muestra la lista real para que ajustes
`PREFERRED_LLMS`.

In [ ]:
from databricks.sdk import WorkspaceClient

PREFERRED_LLMS = [
    "databricks-claude-sonnet-5",
    "databricks-gpt-5-6-luna",
    "databricks-claude-3-7-sonnet",
    "databricks-meta-llama-3-3-70b-instruct",
    "databricks-llama-4-maverick",
    "databricks-gpt-oss-120b",
    "databricks-gpt-oss-20b",
]

w = WorkspaceClient()
available = {e.name for e in w.serving_endpoints.list()}
LLM_ENDPOINT = next((m for m in PREFERRED_LLMS if m in available), None)

assert LLM_ENDPOINT, (
    "Ninguno de los modelos preferidos está disponible en este workspace.\n"
    f"Preferidos : {PREFERRED_LLMS}\n"
    f"Disponibles: {sorted(available)}\n"
    "Edita PREFERRED_LLMS con alguno de los endpoints disponibles de la lista."
)

print(f"LLM seleccionado: {LLM_ENDPOINT}")

## 3. Entrenar el pipeline TF-IDF

Cuatro etapas de SparkML encadenadas:

| Etapa | Qué hace |
|---|---|
| `RegexTokenizer` | Parte el texto en tokens alfanuméricos, en minúsculas |
| `StopWordsRemover` | Elimina palabras vacías (*the*, *and*, *of*…) que no discriminan |
| `HashingTF` | Mapea tokens a un vector de frecuencias de tamaño fijo |
| `IDF` | Pondera cada término por qué tan raro es en el corpus |

La intuición de TF-IDF: una palabra importa si aparece mucho **en este chunk** y poco **en el
resto del corpus**. Es puramente léxica — cuenta palabras, no entiende significado. Esa
limitación es justamente lo que el índice vectorial va a resolver.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF

N_FEATURES = 1 << 14  # 16384

tokenizer = RegexTokenizer(
    inputCol="chunk_text", outputCol="tokens", pattern=r"\W+", toLowercase=True, minTokenLength=2
)
remover = StopWordsRemover(inputCol="tokens", outputCol="tokens_clean")
hashing = HashingTF(inputCol="tokens_clean", outputCol="tf", numFeatures=N_FEATURES)
idf = IDF(inputCol="tf", outputCol="tfidf", minDocFreq=1)

tfidf_pipeline = Pipeline(stages=[tokenizer, remover, hashing, idf])

corpus_df = spark.table(T_CORPUS)
tfidf_model = tfidf_pipeline.fit(corpus_df)
corpus_vec = tfidf_model.transform(corpus_df).select(
    "chunk_id", "doc_id", "title", "doc_type", "chunk_text", "tfidf"
)

tfidf_model.write().overwrite().save(TFIDF_MODEL_PATH)
print(f"Pipeline TF-IDF entrenado sobre {corpus_df.count():,} chunks y guardado en {TFIDF_MODEL_PATH}")

## 4. Matriz de vectores en memoria

Traemos los vectores del corpus al driver como una matriz dispersa de SciPy. Los vectores
TF-IDF son dispersos por naturaleza (cada chunk usa unos pocos cientos de los 16384 términos
posibles), así que el formato CSR es muy compacto y el producto punto es inmediato.

Normalizamos cada fila a norma 1. Con vectores normalizados, la similitud coseno se reduce a
un producto punto — una sola multiplicación matriz-vector para consultar todo el corpus.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

rows = corpus_vec.select("chunk_id", "doc_id", "title", "doc_type", "chunk_text", "tfidf").collect()

CHUNK_IDS = [r["chunk_id"] for r in rows]
CHUNK_META = {
    r["chunk_id"]: {
        "doc_id": r["doc_id"],
        "title": r["title"],
        "doc_type": r["doc_type"],
        "chunk_text": r["chunk_text"],
    }
    for r in rows
}

indptr, indices, data = [0], [], []
for r in rows:
    v = r["tfidf"]
    indices.extend(v.indices.tolist())
    data.extend(v.values.tolist())
    indptr.append(len(indices))

CORPUS_MATRIX = normalize(
    csr_matrix((data, indices, indptr), shape=(len(rows), N_FEATURES)), norm="l2", axis=1
)

print(f"Matriz: {CORPUS_MATRIX.shape[0]:,} chunks x {CORPUS_MATRIX.shape[1]:,} features")
print(f"Densidad: {CORPUS_MATRIX.nnz / (CORPUS_MATRIX.shape[0] * CORPUS_MATRIX.shape[1]) * 100:.4f}%")

## 5. Función de retrieval

Vectoriza la pregunta con **el mismo pipeline** entrenado sobre el corpus (esto es
imprescindible: el mismo hashing y los mismos pesos IDF), y devuelve los `k` chunks con mayor
similitud coseno.

In [ ]:
def retrieve_tfidf(question: str, k: int = 3) -> list[dict]:
    """Devuelve los k chunks más similares a la pregunta, según TF-IDF + coseno."""
    q_df = spark.createDataFrame([(question,)], ["chunk_text"])
    q_vec = tfidf_model.transform(q_df).select("tfidf").collect()[0]["tfidf"]

    q_sparse = normalize(
        csr_matrix(
            (q_vec.values.tolist(), q_vec.indices.tolist(), [0, len(q_vec.indices)]),
            shape=(1, N_FEATURES),
        ),
        norm="l2",
    )

    scores = (CORPUS_MATRIX @ q_sparse.T).toarray().ravel()
    top_idx = np.argsort(-scores)[:k]

    out = []
    for i in top_idx:
        cid = CHUNK_IDS[i]
        out.append({"chunk_id": cid, "score": float(scores[i]), **CHUNK_META[cid]})
    return out

# Prueba rápida
demo_q = spark.table(T_DEV).select("question").first()["question"]
print(f"Pregunta de ejemplo: {demo_q}\n")
for c in retrieve_tfidf(demo_q, k=3):
    print(f"  [{c['score']:.4f}] {c['chunk_id']}  ({c['doc_type']})")
    print(f"          {c['chunk_text'][:120]}...")

## 6. Función de generación

El prompt es el corazón del control de alucinaciones. Tres reglas explícitas:

1. **Responder solo con los chunks dados.** Sin esto, el modelo completa con lo que sabe de
   su entrenamiento — exactamente la alucinación que el benchmark penaliza.
2. **Admitir cuando no está.** Un sistema que nunca dice "no sé" está mintiendo parte del
   tiempo.
3. **Declarar las citas en una línea con formato fijo.** Necesitamos poder parsearlas.

El prompt va en inglés porque el corpus está en inglés y mezclar idiomas entre instrucción y
evidencia degrada la calidad de la respuesta.

In [ ]:
SYSTEM_PROMPT_V1 = """You are a grounded QA assistant. Answer ONLY using the provided chunks.

Rules:
1. If the chunks do not contain the answer, reply exactly: NOT_FOUND
2. Answer in one short sentence, copying key facts verbatim from the chunks.
3. End your reply with a line in exactly this format:
CITED: chunk_id_1, chunk_id_2"""


def build_user_message(question: str, chunks: list[dict]) -> str:
    blocks = [f"[{c['chunk_id']}] {c['chunk_text']}" for c in chunks]
    return "CHUNKS:\n" + "\n\n".join(blocks) + f"\n\nQUESTION: {question}"

### Cliente del LLM con reintentos

El SDK de Databricks expone un cliente compatible con OpenAI apuntando a los endpoints del
workspace. No hace falta gestionar tokens: la autenticación va implícita en el notebook.

Los reintentos con backoff exponencial no son opcional: bajo uso justo, un throttling
transitorio es normal y no debe tumbar una evaluación de 20 preguntas a mitad de camino.

In [ ]:
import re
import time

openai_client = w.serving_endpoints.get_open_ai_client()


def call_llm(system_prompt: str, user_msg: str, max_retries: int = 3) -> str:
    """Llama al LLM con reintentos y backoff exponencial."""
    for attempt in range(max_retries):
        try:
            resp = openai_client.chat.completions.create(
                model=LLM_ENDPOINT,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_msg},
                ],
                max_tokens=300,
                temperature=0.0,  # determinístico: queremos reproducibilidad, no creatividad
            )
            return resp.choices[0].message.content
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt
            print(f"  reintento {attempt + 1}/{max_retries} en {wait}s ({type(e).__name__})")
            time.sleep(wait)


def parse_answer_and_citations(raw: str, retrieved: list[dict]) -> tuple[str, list[str]]:
    """Separa el texto de la respuesta de la línea CITED:, validando los IDs contra lo recuperado."""
    valid_ids = {c["chunk_id"] for c in retrieved}
    match = re.search(r"CITED:\s*(.+?)\s*$", raw, flags=re.MULTILINE | re.DOTALL)

    if match:
        cited = [c.strip() for c in match.group(1).replace("\n", ",").split(",") if c.strip()]
        cited = [c for c in cited if c in valid_ids]  # descarta IDs inventados por el modelo
        answer = raw[: match.start()].strip()
    else:
        # El modelo no emitió la línea. Fallback: citar todo lo recuperado.
        cited = [c["chunk_id"] for c in retrieved]
        answer = raw.strip()

    return answer, cited

## 7. Pipeline RAG completo

In [ ]:
def rag_tfidf(question: str, k: int = 3) -> dict:
    """Pipeline completo: retrieve -> generate -> cite."""
    chunks = retrieve_tfidf(question, k)
    raw = call_llm(SYSTEM_PROMPT_V1, build_user_message(question, chunks))
    answer, cited = parse_answer_and_citations(raw, chunks)
    return {
        "answer": answer,
        "cited_chunk_ids": cited,
        "retrieved_chunk_ids": [c["chunk_id"] for c in chunks],
    }


demo = rag_tfidf(demo_q, k=3)
print(f"Pregunta : {demo_q}")
print(f"Respuesta: {demo['answer']}")
print(f"Citas    : {demo['cited_chunk_ids']}")

## 8. Métricas de la competencia

La métrica exacta de Kaggle no es pública, pero la descripción oficial dice que el puntaje
premia respuestas correctas **con citas precisas**. Estas tres métricas capturan esa idea y
son las mismas que formalizaremos como scorers de MLflow en el notebook 104:

| Métrica | Fórmula | Qué castiga |
|---|---|---|
| **precision** | \|citadas ∩ gold\| / \|citadas\| | Citar de más (ruido) |
| **recall** | \|citadas ∩ gold\| / \|gold\| | Omitir evidencia necesaria |
| **F1** | media armónica | Equilibrio entre ambas |

Añadimos `hit_rate@k`: ¿al menos un chunk gold quedó entre los `k` recuperados? Esta separa
responsabilidades — si `hit_rate` es bajo, el problema es el **retrieval**; si `hit_rate` es
alto pero la precisión es baja, el problema está en el **prompt o la política de citación**.

In [ ]:
def citation_prf(cited: list[str], gold: list[str]) -> dict:
    """Precision, recall y F1 de las citas contra la evidencia gold."""
    c, g = set(cited), set(gold)
    if not g:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    inter = len(c & g)
    p = inter / len(c) if c else 0.0
    r = inter / len(g)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {"precision": p, "recall": r, "f1": f1}


def hit_rate(retrieved: list[str], gold: list[str]) -> float:
    """1.0 si al menos un chunk gold fue recuperado."""
    return 1.0 if set(retrieved) & set(gold) else 0.0

## 9. Evaluar la línea base

Procesamos la muestra secuencialmente con una pausa breve entre llamadas, respetando el uso
justo. Con `N_EVAL=20` esto toma alrededor de un minuto.

In [ ]:
sample = spark.table(T_DEV).orderBy("question_id").limit(N_EVAL).collect()

results = []
for i, row in enumerate(sample, 1):
    out = rag_tfidf(row["question"], k=3)
    gold = list(row["gold_chunk_ids"])
    prf = citation_prf(out["cited_chunk_ids"], gold)
    results.append({
        "question_id": row["question_id"],
        "question": row["question"],
        "answer": out["answer"],
        "cited_chunk_ids": out["cited_chunk_ids"],
        "retrieved_chunk_ids": out["retrieved_chunk_ids"],
        "gold_chunk_ids": gold,
        "precision": prf["precision"],
        "recall": prf["recall"],
        "f1": prf["f1"],
        "hit_rate": hit_rate(out["retrieved_chunk_ids"], gold),
    })
    if i % 5 == 0:
        print(f"  {i}/{len(sample)} preguntas procesadas")
    time.sleep(0.5)

print(f"\nCompletado: {len(results)} preguntas.")

In [ ]:
import pandas as pd

res_pd = pd.DataFrame(results)
summary = res_pd[["precision", "recall", "f1", "hit_rate"]].mean()

print("=" * 46)
print("  LÍNEA BASE TF-IDF (k=3)")
print("=" * 46)
for metric, value in summary.items():
    print(f"  {metric:<12} {value:.3f}")
print("=" * 46)

### Persistir los resultados

Los guardamos en Delta para poder comparar contra el pipeline vectorial más adelante sin
tener que volver a gastar llamadas al LLM.

In [ ]:
(spark.createDataFrame(res_pd)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(T_BASELINE))

display(spark.table(T_BASELINE).select(
    "question_id", "answer", "cited_chunk_ids", "gold_chunk_ids", "precision", "recall", "f1"
))

## 10. Efecto de `k` sobre el retrieval

Esta celda **no llama al LLM**: mide solo la calidad del retrieval variando `k`. Es barata,
así que puedes explorar el rango completo.

> **Antes de ejecutar:** Vas a ver `hit_rate` para k = 1, 3, 5 y 10.
>
> ¿Cómo esperas que se comporte la curva al subir k — crece siempre, se aplana, baja?
> Y la pregunta que importa: si subir k siempre mejora el `hit_rate`, ¿por qué no usar
> k = 50 y terminar el problema?

In [ ]:
k_rows = []
for k in [1, 3, 5, 10]:
    hits = [
        hit_rate([c["chunk_id"] for c in retrieve_tfidf(r["question"], k)], list(r["gold_chunk_ids"]))
        for r in sample
    ]
    # Precisión máxima teórica si se citaran los k recuperados
    max_prec = np.mean([min(len(r["gold_chunk_ids"]), k) / k for r in sample])
    k_rows.append({"k": k, "hit_rate": np.mean(hits), "precision_si_cita_todo": max_prec})

display(pd.DataFrame(k_rows))

La tabla muestra la tensión central del ejercicio: subir `k` mejora el `hit_rate` (más
probable que la evidencia esté ahí) pero hunde la precisión máxima alcanzable si el agente
cita todo lo que recupera. Con k=10 y una sola evidencia gold, el techo de precisión es 0.10.

De ahí que el agente necesite una **política de citación**: recuperar amplio para no perder
evidencia, pero citar estrecho para no diluir la precisión. Eso es exactamente lo que vas a
parametrizar en el notebook 103 y a optimizar en el 105.

## 11. Dónde falla TF-IDF

Antes de pasar a embeddings, conviene ver el modo de fallo concreto. Buscamos preguntas donde
el retrieval no encontró ningún chunk gold.

In [ ]:
fallos = res_pd[res_pd["hit_rate"] == 0.0]
print(f"Preguntas donde el retrieval TF-IDF falló por completo: {len(fallos)} de {len(res_pd)}\n")

for _, r in fallos.head(3).iterrows():
    print(f"Pregunta   : {r['question']}")
    print(f"Recuperado : {r['retrieved_chunk_ids']}")
    print(f"Esperado   : {r['gold_chunk_ids']}")
    gold_txt = CHUNK_META.get(r["gold_chunk_ids"][0], {}).get("chunk_text", "")
    print(f"Texto gold : {gold_txt[:200]}...")
    print("-" * 90)

Si hay fallos, míralos con atención: en general la pregunta y el chunk correcto **hablan de
lo mismo con palabras distintas**. TF-IDF cuenta términos; no sabe que *"outage"* y
*"service disruption"* son parientes, ni que *"cómo revierto un despliegue"* se responde con
un texto que dice *"rollback procedure"*.

Ese es precisamente el vacío que llenan los embeddings: representan el **significado** en un
espacio vectorial donde textos que quieren decir lo mismo quedan cerca, aunque no compartan
una sola palabra.